## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching web, or running code. Tools are pairings of:

    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema).
    2. A function or coroutine to execute.

In [11]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re known for their ability to mimic human speech, right? But why do they do that? Maybe it\'s related to their social behavior. I remember that in the wild, parrots live in flocks, so communication must be important for them. Maybe they mimic sounds to bond with their group or find mates. But how does that translate to talking to humans?\n\nI\'ve heard that parrots have a syrinx, which is their vocal organ. It\'s different from the human larynx. So their ability to mimic sounds is part of their natural behavior. But why would they choose to mimic human speech specifically? Maybe because they\'re around humans a lot in captivity? Or is there a deeper reason related to their intelligence and social learning?\n\nAlso, I think some parrots can associate words with actions or objects. For example, a parrot might learn to say "hello" when someone enters the r

In [12]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather of a location."""
    return f"It's sunny in {location}" #Docstrings are important to let LLMs know what the tool is doing / being used for.

model_with_tools=model.bind_tools([get_weather])


In [ ]:
response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '6vrbahxvp', 'type': 'tool_call'}['name']
Args: {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '6vrbahxvp', 'type': 'tool_call'}['args']


### Tool Eexecution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72 DegF and Sunny"
